# sol_chat LoRA 학습 (Colab)

PC의 `G:\내 드라이브\sol_chat` 폴더를 Colab에서 Google Drive로 그대로 열어서 학습합니다.
코드, 데이터, 결과(어댑터)가 모두 Drive에 있으니 따로 업로드/다운로드할 필요가 없습니다.

**시작 전**: 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU` 선택

순서: GPU 확인 → Drive 연결 → 패키지 설치 → 학습 → 평가(베이스 vs LoRA 비교)

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# PC의 '내 드라이브/sol_chat' = Colab의 /content/drive/MyDrive/sol_chat
%cd /content/drive/MyDrive/sol_chat
!ls

In [ ]:
!pip install -q -r requirements.txt -r requirements-train.txt

## 학습 설정 확인

`configs/train.yaml` 에서 베이스 모델, 데이터 경로, LoRA/학습 하이퍼파라미터를 바꿀 수 있습니다.
- GPU 메모리가 부족하면(Out of memory) `batch_size` 를 줄이거나 `quantization.load_in_4bit: true` 로 바꾸세요.
- 세션이 끊겨도 같은 명령을 다시 실행하면 `outputs/checkpoints/` 에서 이어서 학습합니다.

In [ ]:
!cat configs/train.yaml

In [ ]:
# 처음엔 동작 확인용으로 짧게 돌려보고 싶다면: --max-steps 10 추가
!python -m training.train --config configs/train.yaml

## 평가: 같은 질문에 대한 베이스 vs LoRA 답변

질문 목록: `data/examples/eval_questions.txt`. 결과는 `outputs/eval/` 에 저장됩니다.
예시 데이터로 학습했다면 LoRA 쪽 답변이 반말 + 마지막 줄 `📌 한 줄 요약:` 형식으로 바뀌어 있어야 합니다.

In [ ]:
!python -m training.evaluate --config configs/train.yaml

In [ ]:
import glob, os
from IPython.display import Markdown, display
latest = max(glob.glob('outputs/eval/*.md'), key=os.path.getmtime)
display(Markdown(open(latest, encoding='utf-8').read()))

## 다음 단계

학습된 어댑터는 `outputs/adapters/<run_name>/` 에 있습니다 (PC의 Drive 폴더에도 동기화됨).
웹 화면에서 이 모델과 대화하려면 `notebooks/serve_vllm_colab.ipynb` 를 실행하거나,
PC에서 `configs/model.yaml` 의 `my-lora-local` 항목을 켜세요 (README 참고).